In [1]:
# BM25 is a text-search method used to find relevant documents based on the words in the query.
# Find documents that contain the important words from my question.
"""
Documents
   ↓
tokenize()
   ↓
count words
   ↓
calculate word importance (IDF)
   ↓
BM25 score
   ↓
sort documents
   ↓
top K results
"""
import getpass
import os
import math
import numpy as np
from collections import Counter
from dataclasses import dataclass
import re 
import sys
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")
from llm_config import ollama, MODEL_OLLAMA

In [2]:
# 1. tokenize() — break text into words
documents = [
    "FAISS is a library for vector similarity search.",
    "SQLite is a lightweight relational database.",
    "BM25 is a keyword based text retrieval algorithm.",
    "FAISS can perform fast nearest neighbor search."
]

# Convert string into common words
def tokenizer(input_text:str) -> list[str]:
    text = input_text.lower()
    tokens = re.findall(r'\b\w+\b', text)  # \b Word boundary, \w+ - A letter, number, or _ , + One or more. re.findall means Find all matches.
    """
    "Don't treat \b and \w as Python escape characters. Pass them directly to the regex engine." r means RAW
    \b   \w+   \b
    |    |     |
    start  word  end
    """
    return [ t for t in tokens if len(t)>1]          # in the example "a" from text is removed . python is a easy language

tokenizer("python is a easy language")

<>:14: SyntaxWarning: invalid escape sequence '\w'
<>:14: SyntaxWarning: invalid escape sequence '\w'
C:\Users\allan\AppData\Local\Temp\ipykernel_36416\1323579517.py:14: SyntaxWarning: invalid escape sequence '\w'
  "Don't treat \b and \w as Python escape characters. Pass them directly to the regex engine." r means RAW


['python', 'is', 'easy', 'language']

In [ ]:
@dataclass   
# This is basically telling Python: = "This class is mainly used to store some configuration/data.
# @dataclass automatically creates an __init__() for you based on: k1 and b

def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.__post_init__()

In [ ]:
@dataclass                                          
class BM25:
    k1: float = 1.5                                 # BM25 parametyer - k1 → controls how much repeated words matter
    b: float = 0.75                                 # BM25 parametyer - b  → controls adjustment for document length
    
    def __post_init__(self):                        # A dataclass automatically calls __post_init__() immediately after its generated __init__().
        self.corpus = []
        self.doc_freqs = []
        self.doc_lengths = []
        self.avgdl = 0
        self.idf = {}
        self.n_docs = 0

    def index(self, documents: list[str]):          # BM25 reads all your documents and prepares them for searching. Normal method
        self.corpus = documents
        self.n_docs = len(documents)
        self.doc_freqs = []
        self.doc_lengths = []
        doc_containing_term = Counter()
        
        for doc in documents:
            tokens = tokenizer(doc)
            self.doc_lengths.append(len(tokens))
            freq = Counter(tokens)
            self.doc_freqs.append(freq)
            for term in set(tokens):
                doc_containing_term[term] += 1
        
        self.avgdl = sum(self.doc_lengths) / self.n_docs if self.n_docs > 0 else 0
        self.idf = {}
        for term, n in doc_containing_term.items():
            self.idf[term] = math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1)
    
    def score(self, query: str, doc_idx: int) -> float:
        query_terms = tokenize(query)
        doc_freq = self.doc_freqs[doc_idx]
        doc_len = self.doc_lengths[doc_idx]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = doc_freq.get(term, 0)
            idf = self.idf[term]
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += idf * (numerator / denominator)
        return score
    
    def search(self, query: str, top_k: int = 10) -> list[tuple[int, float]]:
        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return [(idx, score) for idx, score in scores[:top_k] if score > 0]

IndentationError: unindent does not match any outer indentation level (<string>, line 21)